# ContractRisk — candidato Hydrogen, calibración sintética

Runtime CPU. Candidato fijo gemini-4-hydrogen, seleccionado explícitamente tras dos errores 503 y cero notas del modelo anterior. Conserva la escala, anclas, piloto, agenda y umbrales; es un candidato NUEVO, no un reintento oculto ni selección entre notas. No asegura disponibilidad o buen juicio.

Usa GenerateContent según la documentación actual, HIGH thinking y 65536 tokens máximos combinados para evitar el cap bajo desaconsejado por el fabricante. JSON Schema, sin candidateCount ni parámetros de sampling ignorados. Sin herramientas, firmas o razonamiento privado guardado. No habilita facturación ni hace fallback. Si falla API/cuota/formato o el primer juicio, se detiene.

Mismos secretos GEMINI_API_KEY y WANDB_API_KEY y misma cuenta/carpeta de Drive. Deben seguir intactas las dos reservas de USD 0,3072 (USD 0,6144 en conjunto) y las marcas anteriores. No son cargos confirmados. El NUEVO candidato fija antes de observar sus notas un sublímite acumulado de estimación de USD 2 (incluidas las reservas anteriores), dentro del techo total autorizado USD 5. No activa pago: confirmar que el proyecto sigue Free Tier sin facturación.

Antes de generar verifica catálogo y conteo. El catálogo por sí solo no prueba acceso de inferencia. El primer juicio sintético es también la prueba real de servicio, no hay un ping generativo extra. Solo si pasan las nueve anclas piloto completa 38 juicios. Confirmación nueva y auditoría humana quedan pendientes incluso si pasa. No modifica main ni Space.


In [ ]:
import os, sys, json, subprocess, tempfile
from pathlib import Path
from google.colab import userdata

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_hydrogen_judge_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
# Required by historical-integrity unit tests; no inference from that commit is rerun.
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin',
    '6ec8dcdf22433c632b5eb0e5539401c33b9c923d'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', '71912f1c5755790381a8e39c8a67801bcbf9243d'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', '7afdda0212bf38ad1accd043a968db722569c618'], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)
for name in ['GEMINI_API_KEY', 'WANDB_API_KEY']:
    try:
        value = userdata.get(name)
        if not value:
            raise ValueError('Empty secret')
        os.environ[name] = value
        del value
    except Exception:
        raise RuntimeError('Habilita el acceso de ESTE notebook al secreto ' + name + '. No pegues la clave.') from None

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError('Proceso detenido; conservar resultados y consultar ' + logfile)


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'final/requirements-api-judge.txt'], 'installation.log')
run_visible([sys.executable, '-c', "from pathlib import Path; from final.run_judge_calibration import verify_review; verify_review(Path('final/eval/calibration_review_v1')); print('Anclas aprobadas y hashes comprobados.')"], 'review_check.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUDGET_DIR = Path('/content/drive/MyDrive/ContractRisk_final_budget')
confirmation = input('Confirma que el proyecto de GEMINI_API_KEY es Free Tier y NO tiene facturación activada. Escribe GRATUITO para continuar: ')
assert confirmation.strip() == 'GRATUITO', 'No se hará ninguna petición API sin confirmar el proyecto gratuito.'
del confirmation
print('Registro persistente de presupuesto en Drive. No borrar para repetir corridas.')


In [ ]:
RESULTS = ROOT / 'hydrogen_calibration_results'
run_visible([sys.executable, '-u', 'final/run_hydrogen_calibration.py', '--output', str(RESULTS),
    '--budget-dir', str(BUDGET_DIR), '--confirm-free-project', '--require-wandb'], 'hydrogen_calibration.log')


## Interpretación

Si se detiene, descargar evidencia; no borrar marcas/registro Drive, activar facturación, cambiar modelo o repetir automáticamente. Las cuotas/disponibilidad no se deducen de la lista del Playground.

Las notas son brutas; no se convierten errores en 1 ni aciertos en 4. Un error técnico sin nota no es mala puntuación. Una regresión aprobada aún requiere auditoría humana y confirmación nueva. El consumo guardado es una estimación conservadora de uso/reserva, no una factura.


In [ ]:
RESULTS = ROOT / 'hydrogen_calibration_results'
for name in ['metadata.json', 'pilot_summary.json', 'regression_summary.json', 'partial_regression_summary.json']:
    path = RESULTS / name
    if path.exists():
        print(name, path.read_text(encoding='utf-8'))


In [ ]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for logfile in ROOT.glob('*.log'):
    shutil.copy2(logfile, RESULTS / logfile.name)
for name in ['hydrogen_calibration_config.json', 'hydrogen_budget_policy.json', 'api_budget_policy.json']:
    shutil.copy2(ROOT / 'final' / name, RESULTS / name)
reviewed = RESULTS / 'reviewed_calibration_pack'
if not reviewed.exists():
    shutil.copytree(ROOT / 'final/eval/calibration_review_v1', reviewed)
package = shutil.make_archive(str(ROOT / 'contractrisk_final_juez_hydrogen_resultados'), 'zip', RESULTS)
files.download(package)
